# Diagnostic 1 — score vs kernel alignment, per event (interactive)

Each line is **one event**: the model's score as the kernel slides across the merger.

- x-axis `e` = (kernel right-edge time) − (coalescence). `e = 0` → kernel ends at the merger; `e < 0` → pre-merger.
- **red = signal** (a loud injection) — should **dip in σ / peak in −σ at slightly negative `e`** (e.g. ~−3 for the id11 3 s-before model).
- **blue = background** (no signal) — should stay **flat and uncertain**.

**Hover a line to highlight it** (the rest dim out). Click legend entries to show/hide.

Generate the CSV first with `scripts/diag_score_timeseries.py` (needs a GPU). This notebook is pure plotting.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

CSV = "/n/holystore01/LABS/iaifi_lab/Lab/kyoon/aframe_linoss/runs/regression_sv/diag/premerger_59-60s_ft/score_timeseries.csv"
Y_COL = "neg_sigma"   # what to plot on y: 'neg_sigma' (detection statistic), 'chirp_sigma', or 'chirp_mean'
TRAINED_E = -3.0      # model's trained kernel offset = -(training window_offset); id11/59-60s -> -3.0

df = pd.read_csv(CSV)
print(f"{df.event_id.nunique()} events, {len(df)} points")
df.head()

In [ ]:
SIGNAL_RGB, BG_RGB = "214,39,40", "31,119,180"   # red, blue
rgb_of = {}   # trace index -> 'r,g,b' so we can restyle alpha on hover

fig = go.FigureWidget()
for eid, g in df.groupby("event_id"):
    g = g.sort_values("e")
    is_signal = g["kind"].iloc[0] == "signal"
    rgb = SIGNAL_RGB if is_signal else BG_RGB
    snr = g["snr"].iloc[0]
    name = f"{eid} (SNR {snr:.0f})" if is_signal else eid
    fig.add_scatter(
        x=g["e"], y=g[Y_COL], mode="lines+markers", name=name,
        legendgroup=g["kind"].iloc[0],
        line=dict(color=f"rgba({rgb},0.35)", width=1.5),
        marker=dict(size=4, color=f"rgba({rgb},0.35)"),
    )
    rgb_of[len(fig.data) - 1] = rgb

fig.add_vline(x=0, line=dict(color="gray", dash="dot"),
              annotation_text="merger in kernel (OOD)", annotation_position="top")
fig.add_vline(x=TRAINED_E, line=dict(color="green", dash="dash"),
              annotation_text="trained offset", annotation_position="top")
fig.update_layout(
    template="plotly_white", height=620, hovermode="closest",
    xaxis_title="e = kernel right-edge − coalescence [s]   (0 = kernel ends at merger; <0 = pre-merger)",
    yaxis_title=Y_COL,
    title="Score vs kernel alignment — hover a line to highlight",
)

def _style(i, alpha, width):
    col = f"rgba({rgb_of[i]},{alpha})"
    fig.data[i].line.width = width
    fig.data[i].line.color = col
    fig.data[i].marker.color = col

def _highlight(trace, points, state):
    h = points.trace_index
    with fig.batch_update():
        for i in rgb_of:
            _style(i, 1.0, 4) if i == h else _style(i, 0.07, 1)

def _reset(trace, points, state):
    with fig.batch_update():
        for i in rgb_of:
            _style(i, 0.35, 1.5)

for tr in fig.data:
    if hasattr(tr, "on_hover"):
        tr.on_hover(_highlight)
        tr.on_unhover(_reset)
fig

**Reading it:**
- Red (signal) should separate from blue (background): in `−σ`, red rises above blue across the pre-merger range `e ∈ [−4, −1]`; blue stays flat.
- **Two different "best" points.** The model's *confidence* (largest `−σ`) keeps growing toward the merger because the late inspiral is louder — but its chirp-mass *accuracy* is only trustworthy at the **green line (trained offset, e≈−3)**. Right at the **gray line (e=0)** the merger enters the short kernel: that's out-of-distribution, so the model gets *overconfident but wrong*. This is why Diagnostic 2 fixes `e` at the trained offset instead of taking the most-confident alignment.
- If red and blue overlap everywhere, the model isn't distinguishing signal from noise at that SNR — lower `--snr-min`, or set `Y_COL = "chirp_sigma"` to see raw uncertainty / `"chirp_mean"` to watch the predicted chirp mass settle near the trained offset.

**Reading it:** if the red curves rise into a clear bump at slightly negative `e` while the blue curves stay low and noisy, the model is localizing signals in time and the detection statistic separates signal from background. The `e` of the red peak tells you the model's effective pre-merger offset. If red and blue overlap, the model isn't distinguishing them at that SNR — lower `--snr-min` when generating, or switch `Y_COL` to `chirp_sigma` to see the raw uncertainty.